In [54]:
import pandas as pd
import numpy as np
# from tqdm import tqdm
import random
import time
import csv
import json
import openai
import os
from tqdm import tqdm
import time
import ast

In [55]:
baseline_results = pd.read_csv("results/baseline_gpt_oss_1000.csv") #change name to baseline_gpt_oss_1000
baseline_results

,Argument ID,values,seconds
0,A01001,"{'Be creative':0, 'Be curious':0, 'Have freedo...",30.924304
1,A01002,"{'Be creative':1, 'Be curious':0, 'Have freedo...",25.552757
2,A01003,"{'Be creative':0, 'Be curious':0, 'Have freedo...",32.587301
3,A01004,"{'Be creative':0, 'Be curious':0, 'Have freedo...",21.698446
4,A01005,"{'Be creative':0, 'Be curious':0, 'Have freedo...",35.828496
5,A01006,"{'Be creative':0, 'Be curious':0, 'Have freedo...",33.169319
6,A01007,"{'Be creative':0, 'Be curious':0, 'Have freedo...",29.317023
7,A01008,"{'Be creative':0, 'Be curious':0, 'Have freedo...",28.456753
8,A01009,"{'Be creative':0, 'Be curious':0, 'Have freedo...",36.537273
9,A01010,"{'Be creative':0, 'Be curious':0, 'Have freedo...",39.077288


In [56]:
baseline_results['values'] = baseline_results['values'].apply(ast.literal_eval) # transformamos el output del LLM a un diccionario
baseline_results

,Argument ID,values,seconds
0,A01001,"{'Be creative': 0, 'Be curious': 0, 'Have free...",30.924304
1,A01002,"{'Be creative': 1, 'Be curious': 0, 'Have free...",25.552757
2,A01003,"{'Be creative': 0, 'Be curious': 0, 'Have free...",32.587301
3,A01004,"{'Be creative': 0, 'Be curious': 0, 'Have free...",21.698446
4,A01005,"{'Be creative': 0, 'Be curious': 0, 'Have free...",35.828496
5,A01006,"{'Be creative': 0, 'Be curious': 0, 'Have free...",33.169319
6,A01007,"{'Be creative': 0, 'Be curious': 0, 'Have free...",29.317023
7,A01008,"{'Be creative': 0, 'Be curious': 0, 'Have free...",28.456753
8,A01009,"{'Be creative': 0, 'Be curious': 0, 'Have free...",36.537273
9,A01010,"{'Be creative': 0, 'Be curious': 0, 'Have free...",39.077288


#### Error Checking


Check if the predictions have been made correctly (dictionary with 54 keys and 54 values that are integers 0 / 1)

In [57]:
expected_keys_in_order = [
    'Be creative', 'Be curious', 'Have freedom of thought', 'Be choosing own goals',
    'Be independent', 'Have freedom of action', 'Have privacy', 'Have an exciting life',
    'Have a varied life', 'Be daring', 'Have pleasure', 'Be ambitious', 'Have success',
    'Be capable', 'Be intellectual', 'Be courageous', 'Have influence',
    'Have the right to command', 'Have wealth', 'Have social recognition',
    'Have a good reputation', 'Have a sense of belonging', 'Have good health',
    'Have no debts', 'Be neat and tidy', 'Have a comfortable life',
    'Have a safe country', 'Have a stable society', 'Be respecting traditions',
    'Be holding religious faith', 'Be compliant', 'Be self-disciplined',
    'Be behaving properly', 'Be polite', 'Be honoring elders', 'Be humble',
    'Have life accepted as is', 'Be helpful', 'Be honest', 'Be forgiving',
    'Have the own family secured', 'Be loving', 'Be responsible',
    'Have loyalty towards friends', 'Have equality', 'Be just',
    'Have a world at peace', 'Be protecting the environment',
    'Have harmony with nature', 'Have a world of beauty', 'Be broadminded',
    'Have the wisdom to accept others', 'Be logical', 'Have an objective view'
]

def is_valid(d):
    return (
        isinstance(d, dict)
        and len(d) == 54 # se han predicho 54 valores
        and list(d.keys()) == expected_keys_in_order # se cumple el orden y los nombres de los valores presentes en la lista original del prompt
        and all(isinstance(v, (int)) for v in d.values()) # es un número entero
        and all(v in (0, 1) for v in d.values()) # los valores son 0 o 1
    )

valid_mask = baseline_results["values"].apply(is_valid)

all_valid = valid_mask.all()

invalid_rows = baseline_results[~valid_mask]

# print all rows from the dataframe that do not follow the specified rules 
print(len(invalid_rows), "arguments have not been predicted according to the specified rules (54 keys), 54 values of 0/1)")

5 arguments have not been predicted according to the specified rules (54 keys), 54 values of 0/1)


In [58]:
invalid_rows

,Argument ID,values,seconds
5,A01006,"{'Be creative': 0, 'Be curious': 0, 'Have free...",33.169319
9,A01010,"{'Be creative': 0, 'Be curious': 0, 'Have free...",39.077288
16,A01017,"{'Be creative': 0, 'Be curious': 0, 'Have free...",21.583027
20,A02001,{'Be creative - allowing for more creativity o...,42.976105
43,A03004,"{'Be creative': 0, 'Be curious': 0, 'Have free...",28.955575


In [ ]:
I = 0
for bad_prediction in invalid_rows["values"]:
    print("\n\nInstancia mal clasificada:\n", bad_prediction)
    print(len(bad_prediction) == 54)
    print(list(bad_prediction.keys()) == expected_keys_in_order)
    print(all(isinstance(v, (int)) for v in bad_prediction.values())) # es un número entero
    print(all(v in (0, 1) for v in bad_prediction.values())) # los valores son 0 o 1

    dict_keys = set(bad_prediction.keys())
    expected_keys = set(expected_keys_in_order)

    missing = expected_keys - dict_keys
    extra = dict_keys - expected_keys



Instancia mal clasificada:
 {'Be creative': 0, 'Be curious': 0, 'Have freedom of thought': 0, 'Be choosing own goals': 0, 'Be independent': 0, 'Have freedom of action': 0, 'Have privacy': 0, 'Have an excited life': 0, 'Have a varied life': 0, 'Be daring': 0, 'Have pleasure': 0, 'Be ambitious': 0, 'Have success': 0, 'Be capable': 0, 'Be intellectual': 0, 'Be courageous': 1, 'Have influence': 1, 'Have the right to command': 0, 'Have wealth': 0, 'Have social recognition': 0, 'Have a good reputation': 0, 'Have a sense of belonging': 0, 'Have good health': 0, 'Have no debts': 0, 'Be neat and tidy': 0, 'Have a comfortable life': 0, 'Have a safe country': 0, 'Have a stable society': 1, 'Be respecting traditions': 0, 'Be holding religious faith': 0, 'Be compliant': 0, 'Be self-disciplined': 0, 'Be behaving properly': 0, 'Be polite': 0, 'Be honoring elders': 0, 'Be humble': 0, 'Have life accepted as is': 0, 'Be helpful': 0, 'Be honest': 0, 'Be forgiving': 0, 'Have the own family secured': 0, 

### Errors

Your dictionary contains:

'Have an excited life'

But the expected list contains:

'Have an exciting life'


------------------------------

Your dictionary contains:

'Have sense of belonging'

Expected list contains:

'Have a sense of belonging'

--------------------------------

